# Fine-Tuning-Style Baseline And Limits

This notebook uses real provider calls with few-shot style examples, but no retrieved policy context. That makes the behavior-adaptation effect visible without running a real fine-tuning job.


## What This Proxy Can And Cannot Show

This notebook uses few-shot examples because they make behavior steering visible without running a training job. That is useful for strategy work: participants can see whether examples help tone, refusal style, or answer shape before deciding that real fine-tuning is worth the cost.

The proxy has hard limits. Few-shot examples are sent with one request and disappear afterward. They do not prove that a trained adapter would generalize, preserve safety behavior, or avoid regressions on unrelated tasks. Real fine-tuning still needs a held-out evaluation set and a rollback plan.


## Fine-Tuning Concepts Without Training Yet

**Supervised fine-tuning** teaches a model from examples of desired inputs and outputs. **Parameter-efficient fine-tuning** methods such as LoRA add or update a smaller set of parameters instead of changing every base-model weight. These approaches can improve repeated behavior, but they require careful evaluation because the model may also learn unwanted shortcuts.

In this strategy repo, we do not train. Instead, we use few-shot prompting as a low-cost probe. If examples do not help even in the prompt, real fine-tuning is unlikely to be the first thing to try. If examples help but the behavior is still inconsistent across held-out cases, that is evidence that a later fine-tuning unit might be worth exploring.


### Exercise

Do few-shot examples persist as model behavior after the request is finished?


### Solution

No. Few-shot examples condition only the current request; they do not update model weights or adapters.


## Visual Map: Behavior Adaptation Without Fresh Evidence

This notebook uses few-shot style examples as a live-call proxy for behavior adaptation. The visuals below show what real parameter-efficient fine-tuning would change later, while the code here deliberately does not train weights or adapters.

**LoRA adapter mechanism.** Source: [Hugging Face PEFT LoRA diagram](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/lora_diagram.png). This visual shows how an adapter can change model behavior without updating every base-model parameter.

<img src="assets/hf-lora-diagram.png" alt="LoRA adapter diagram" style="max-width: 760px; width: 100%; height: auto;">

---

The next visual shows a different parameter-efficient control mechanism, which helps avoid treating all adaptation methods as the same thing.

**Prefix tuning mechanism.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/prefix-tuning.png). This visual gives another example of learned behavior control, which is different from retrieving fresh policy context at answer time.

<img src="assets/hf-prefix-tuning.png" alt="Prefix tuning diagram" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

Do LoRA or prefix tuning automatically provide fresh policy evidence at answer time?


### Solution

No. They can adapt model behavior, but fresh evidence still requires retrieval or another external context source.


## Load Dependencies And Shared Data

The setup is split into small cells so each responsibility is visible: imports first, then `.env` loading and fixture loading.


In [ ]:
import json
import os
from pathlib import Path
from pprint import pprint
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError

The notebooks share one small support-policy fixture. Keeping it shared makes the strategy comparisons fair: each notebook sees the same documents, requests, and style examples.


In [ ]:
# Load optional provider settings from .env without overwriting shell variables.
load_dotenv(dotenv_path=Path(".env"), override=False)

DATA_PATH = Path("data") / "strategy_cases.json"
with DATA_PATH.open(encoding="utf-8") as file:
    DATA = json.load(file)

# Keep each fixture role explicit so later cells do not mix evidence, examples, and evaluation.
DOCUMENTS = DATA["documents"]
REQUESTS = DATA["support_requests"]
STYLE_EXAMPLES = DATA["style_examples"]
SCENARIOS = DATA["strategy_scenarios"]

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, and {len(STYLE_EXAMPLES)} style examples."
)

## Configure The Provider

The notebooks use the OpenAI-compatible client style. Groq is tried first by default; Mistral is the fallback when the preferred provider has no key.


In [ ]:
class ProviderConfig(BaseModel):
    """Provider settings for one OpenAI-compatible chat endpoint."""

    provider: str
    model: str
    base_url: str
    api_key: str | None = None
    dry_run: bool = True

Provider selection is intentionally explicit. This makes fallback behavior easy to inspect before any model call happens.


In [ ]:
def provider_order(preferred_provider: str | None = None) -> list[str]:
    """Return the preferred provider followed by the other supported provider."""
    requested = (preferred_provider or os.getenv("LLM_PROVIDER") or "groq").lower()

    if requested == "groq":
        return ["groq", "mistral"]
    if requested == "mistral":
        return ["mistral", "groq"]

    # Unknown values keep dry mode readable instead of failing during setup.
    return [requested]

The loader returns dry mode when no key is available. Dry mode is part of the teaching design: notebooks stay runnable during review and in classrooms without credentials.


Each provider has a tiny environment-to-config adapter. Keeping this separate makes the fallback loader shorter and easier to read.


In [ ]:
def provider_config_from_env(provider_name: str) -> ProviderConfig | None:
    """Return live provider settings when that provider has an API key."""
    if provider_name == "groq" and os.getenv("GROQ_API_KEY"):
        return ProviderConfig(
            provider="groq",
            model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
            base_url="https://api.groq.com/openai/v1",
            api_key=os.getenv("GROQ_API_KEY"),
            dry_run=False,
        )

    if provider_name == "mistral" and os.getenv("MISTRAL_API_KEY"):
        return ProviderConfig(
            provider="mistral",
            model=os.getenv("MISTRAL_MODEL", "mistral-small-latest"),
            base_url="https://api.mistral.ai/v1",
            api_key=os.getenv("MISTRAL_API_KEY"),
            dry_run=False,
        )

    return None

The loader checks providers in order and falls back to dry mode when no key is available.


In [ ]:
def load_provider_config(preferred_provider: str | None = None) -> ProviderConfig:
    """Load live provider settings, or return a deterministic dry-run config."""
    # Check providers in priority order so fallback behavior is deterministic.
    for provider_name in provider_order(preferred_provider):
        config = provider_config_from_env(provider_name)
        if config is not None:
            return config

    # No key was found, so keep the notebook runnable without network calls.
    fallback_model = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
    return ProviderConfig(
        provider=provider_order(preferred_provider)[0],
        model=fallback_model,
        base_url="dry-run",
    )

In [ ]:
CONFIG = load_provider_config()
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

if CONFIG.dry_run:
    print("Set GROQ_API_KEY or MISTRAL_API_KEY in .env to run live model calls.")

## Call The Model Or Use Dry Mode

The next cells isolate client creation, request assembly, and fallback behavior. That keeps provider plumbing separate from the lesson logic.


In [ ]:
def get_client(config: ProviderConfig) -> OpenAI | None:
    """Create a client only when a live API key is configured."""
    if config.dry_run or config.api_key is None:
        return None

    return OpenAI(api_key=config.api_key, base_url=config.base_url)

Request parameters are built in one small helper so response formatting can be added only where a notebook needs structured JSON.


In [ ]:
def completion_kwargs(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> dict[str, Any]:
    """Build provider request arguments for chat completions."""
    kwargs: dict[str, Any] = {
        "model": CONFIG.model,
        "messages": messages,
        "temperature": temperature,
    }

    # JSON mode is optional because not every provider/model combination needs it.
    if response_format is not None:
        kwargs["response_format"] = response_format

    return kwargs

`call_model` is the single gateway used by lesson cells. If live credentials are missing or a live call fails, the notebook returns a deterministic teaching response.


Live completion execution is separated from dry-mode routing so the failure handling is easy to follow.


In [ ]:
def live_completion_content(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> str:
    """Run the live chat completion and return message content."""
    client = get_client(CONFIG)
    assert client is not None

    completion = client.chat.completions.create(
        **completion_kwargs(
            messages, temperature=temperature, response_format=response_format
        )
    )
    return completion.choices[0].message.content or ""

`call_model` now only decides between dry output, live output, and live-call fallback.


In [ ]:
def call_model(
    messages: list[dict[str, str]],
    *,
    dry_response: str,
    temperature: float = 0.2,
    response_format: dict[str, str] | None = None,
) -> str:
    """Call the configured provider, or return deterministic dry output."""
    # Dry mode is the expected path for participants without provider credentials.
    if CONFIG.dry_run:
        return dry_response

    try:
        return live_completion_content(
            messages, temperature=temperature, response_format=response_format
        )
    except Exception as error:
        print(f"Live provider call failed: {error}")
        print("Using deterministic dry response so the notebook remains runnable.")
        return dry_response

## Build Few-Shot Style Messages

Few-shot messages make behavior examples visible in the prompt. They do not update model weights.


In [ ]:
def style_messages(examples: list[dict[str, str]]) -> list[dict[str, str]]:
    """Convert approved examples into alternating user and assistant messages."""
    messages: list[dict[str, str]] = []

    for example in examples:
        # The user turn shows the incoming request pattern.
        messages.append({"role": "user", "content": example["user"]})
        # The assistant turn shows the response style we want to imitate.
        messages.append({"role": "assistant", "content": example["assistant"]})

    return messages


def compact_style_examples(examples: list[dict[str, str]]) -> str:
    """Create a readable style summary for prompts and reports."""
    lines = []
    for example in examples:
        lines.append(
            f"- intent={example['intent']} tone={example['tone']}: {example['assistant']}"
        )
    return "\n".join(lines)

In [ ]:
# Exercise: list the unique tones represented in the examples.
# unique_tones = ...


In [ ]:
# Solution
unique_tones = sorted({example["tone"] for example in STYLE_EXAMPLES})
print(unique_tones)

## Build Few-Shot Style Messages

Few-shot prompting is not fine-tuning, but it lets participants observe how examples can steer tone, refusal style, and response shape.


In [ ]:
few_shot_messages = style_messages(STYLE_EXAMPLES)
print(f"Few-shot message count: {len(few_shot_messages)}")
print(compact_style_examples(STYLE_EXAMPLES))

In [ ]:
# Exercise: count how many chat messages the style examples create.
# few_shot_message_count = ...


In [ ]:
# Solution
few_shot_message_count = len(few_shot_messages)
print(few_shot_message_count)

## Call The Model Without Fresh Policy Context

The model receives style examples, but not policy documents. This intentionally tests the limitation: behavior can improve while factual grounding remains unproven.


## Reading Style-Only Outputs

Style-only outputs should be read with two questions in mind. First, did the answer follow the expected behavior pattern: concise, operational, careful refusal, or coaching? Second, did it claim policy facts that were not supplied in the prompt? A good style-only answer can still be incomplete because style examples do not carry source-of-truth policy evidence.

This is why the structured output includes `needs_policy_evidence`. That field is not a perfect judge, but it makes the limitation visible. When the model says it still needs policy evidence, it is admitting that behavior conditioning alone is not enough for grounded support answers.


## Examples Of Style-Only Success And Failure

A style-only answer succeeds when the task is mostly about response behavior. Example: “Reply in a concise operational tone and ask for the missing blocker.” The model can follow that pattern from examples.

A style-only answer fails when it needs current evidence. Example: “Can this duplicate payment be refunded?” The model may imitate the refund-response style, but it still needs the current refund policy to avoid inventing eligibility rules. This distinction is the reason strategy reviews separate behavior quality from grounding quality.


The style-only schema separates tone imitation from evidence needs. This is the central limitation of the notebook.


In [ ]:
class StyleOnlyAnswer(BaseModel):
    """Structured output for style-conditioned answers without retrieval."""

    answer: str
    predicted_tone: str
    needs_policy_evidence: bool

Build the style-only prompt from examples and ask for structured JSON. Notice that no policy context is supplied.


Build the style-only prompt in a helper so it is clear that no policy context is included.


In [ ]:
def style_only_messages(question: str) -> list[dict[str, str]]:
    """Create a few-shot style prompt without retrieved evidence."""
    return [
        {
            "role": "system",
            "content": "Use the style of the examples. Return JSON with answer, predicted_tone, and needs_policy_evidence.",
        },
        *few_shot_messages,
        {"role": "user", "content": question},
    ]

The dry output intentionally says policy evidence is still needed.


In [ ]:
dry_style_only = json.dumps(
    {
        "answer": "Ask for the invoice number and both payment references in a concise support tone, but verify the current refund policy before giving exact eligibility details.",
        "predicted_tone": "concise",
        "needs_policy_evidence": True,
    }
)

Validate the structured style-only response before using it in comparisons.


In [ ]:
def style_only_answer(question: str) -> StyleOnlyAnswer:
    """Ask the provider to answer from style examples without policy context."""
    raw = call_model(
        style_only_messages(question),
        dry_response=dry_style_only,
        response_format={"type": "json_object"},
    )

    try:
        return StyleOnlyAnswer.model_validate_json(raw)
    except ValidationError:
        return StyleOnlyAnswer(
            answer=raw, predicted_tone="unknown", needs_policy_evidence=True
        )

Run a few requests to show that style conditioning can be consistent while still needing policy evidence.


In [ ]:
for request in REQUESTS[:3]:
    print()
    print(request["request_id"], request["question"])
    pprint(style_only_answer(request["question"]).model_dump())

### Exercise

Why does the style-only answer still say policy evidence is needed?


### Solution

The examples teach tone and response shape, but they do not guarantee the current refund, MFA, or rate-limit policy facts.


## Compare What This Can And Cannot Solve

A style-conditioned call can learn the response pattern, but it still cannot prove the current refund window, rate limit, or MFA rule unless those facts are present in the prompt or already reliably known.


In [ ]:
# Summarize tone and evidence needs across the held-out requests.
summary_rows = []
for request in REQUESTS:
    output = style_only_answer(request["question"])
    summary_rows.append(
        {
            "request_id": request["request_id"],
            "expected_tone": request["preferred_tone"],
            "predicted_tone": output.predicted_tone,
            "needs_policy_evidence": output.needs_policy_evidence,
        }
    )
pprint(summary_rows)

In [ ]:
fine_tuning_boundary = {
    "what_we_test_here": "real model behavior under few-shot style conditioning",
    "what_we_do_not_test_here": "whether a trained LoRA or hosted fine-tuned model generalizes",
    "next_repo": "Fine-Tuning Small Models Locally + Evaluation",
}
pprint(fine_tuning_boundary)